# 类继承关系
```mermaid
classDiagram
    class MetaStatsBuilderMixin {
        <<metaclass>>
    }
    class MetaIndicatorBase {
        <<metaclass>>
    }
    class MetaPlotsBuilderMixin {
        <<metaclass>>
    }
    class Wrapping
    class StatsBuilderMixin
    class PlotsBuilderMixin
    class IndicatorBase
    class IndicatorFactory
    class SignalFactory
    
    %% 继承关系
    MetaStatsBuilderMixin <|-- MetaIndicatorBase : metaclass
    MetaPlotsBuilderMixin <|-- MetaIndicatorBase : metaclass

    MetaStatsBuilderMixin <|-- StatsBuilderMixin : metaclass
    MetaPlotsBuilderMixin <|-- PlotsBuilderMixin : metaclass

    Wrapping <|-- IndicatorBase
    StatsBuilderMixin <|-- IndicatorBase
    MetaIndicatorBase <|-- IndicatorBase : metaclass
    PlotsBuilderMixin <|-- IndicatorBase
    IndicatorFactory <|-- SignalFactory
```

# class SignalFactory
用于构建信号生成器的工厂类。

## `__init__`
参数
- `*args`：传递给父类 IndicatorFactory 的位置参数
- `mode`：信号生成模式，可选值：
  - `FactoryMode.Entries`：仅生成入场信号
  - `FactoryMode.Exits`：仅生成出场信号  
  - `FactoryMode.Both`：同时生成入场和出场信号
  - `FactoryMode.Chain`：链式处理模式，基于输入入场信号生成新的入场和出场信号
- `input_names`：输入数据名称列表，用于标识传递给信号函数的输入数据
- `attr_settings`：属性设置字典，用于配置输出属性的数据类型等
- `**kwargs`：传递给父类 IndicatorFactory 的关键字参数

### 源码

```python
    def __init__(self,
                 *args,
                 mode: tp.Union[str, int] = FactoryMode.Both,
                 input_names: tp.Optional[tp.Sequence[str]] = None,
                 attr_settings: tp.KwargsLike = None,
                 **kwargs) -> None:
        mode = map_enum_fields(mode, FactoryMode)
        if input_names is None:
            input_names = []
        else:
            input_names = list(input_names)
        if attr_settings is None:
            attr_settings = {}

        if 'entries' in input_names:
            raise ValueError("entries cannot be used in input_names")
        if 'exits' in input_names:
            raise ValueError("exits cannot be used in input_names")
        if mode == FactoryMode.Entries:
            output_names = ['entries']
        elif mode == FactoryMode.Exits:
            input_names = ['entries'] + input_names
            output_names = ['exits']
        elif mode == FactoryMode.Both:
            output_names = ['entries', 'exits']
        else:
            input_names = ['entries'] + input_names
            output_names = ['new_entries', 'exits']
        if 'entries' in input_names:
            attr_settings['entries'] = dict(dtype=np.bool_)
        for output_name in output_names:
            attr_settings[output_name] = dict(dtype=np.bool_)

        IndicatorFactory.__init__(
            self,
            *args,
            input_names=input_names,
            output_names=output_names,
            attr_settings=attr_settings,
            **kwargs
        )
        self.mode = mode

        def plot(_self,
                 entry_y: tp.Optional[tp.ArrayLike] = None,
                 exit_y: tp.Optional[tp.ArrayLike] = None,
                 entry_types: tp.Optional[tp.ArrayLikeSequence] = None,
                 exit_types: tp.Optional[tp.ArrayLikeSequence] = None,
                 entry_trace_kwargs: tp.KwargsLike = None,
                 exit_trace_kwargs: tp.KwargsLike = None,
                 fig: tp.Optional[tp.BaseFigure] = None,
                 **kwargs) -> tp.BaseFigure:  # pragma: no cover
            if _self.wrapper.ndim > 1:
                raise TypeError("Select a column first. Use indexing.")

            if entry_trace_kwargs is None:
                entry_trace_kwargs = {}
            if exit_trace_kwargs is None:
                exit_trace_kwargs = {}
            entry_trace_kwargs = merge_dicts(
                dict(name="New Entry" if mode == FactoryMode.Chain else "Entry"),
                entry_trace_kwargs
            )
            exit_trace_kwargs = merge_dicts(
                dict(name="Exit"),
                exit_trace_kwargs
            )
            if entry_types is not None:
                entry_types = np.asarray(entry_types)
                entry_trace_kwargs = merge_dicts(dict(
                    customdata=entry_types,
                    hovertemplate="(%{x}, %{y})<br>Type: %{customdata}"
                ), entry_trace_kwargs)
            if exit_types is not None:
                exit_types = np.asarray(exit_types)
                exit_trace_kwargs = merge_dicts(dict(
                    customdata=exit_types,
                    hovertemplate="(%{x}, %{y})<br>Type: %{customdata}"
                ), exit_trace_kwargs)
            if mode == FactoryMode.Entries:
                fig = _self.entries.vbt.signals.plot_as_entry_markers(
                    y=entry_y, trace_kwargs=entry_trace_kwargs, fig=fig, **kwargs)
            elif mode == FactoryMode.Exits:
                fig = _self.entries.vbt.signals.plot_as_entry_markers(
                    y=entry_y, trace_kwargs=entry_trace_kwargs, fig=fig, **kwargs)
                fig = _self.exits.vbt.signals.plot_as_exit_markers(
                    y=exit_y, trace_kwargs=exit_trace_kwargs, fig=fig, **kwargs)
            elif mode == FactoryMode.Both:
                fig = _self.entries.vbt.signals.plot_as_entry_markers(
                    y=entry_y, trace_kwargs=entry_trace_kwargs, fig=fig, **kwargs)
                fig = _self.exits.vbt.signals.plot_as_exit_markers(
                    y=exit_y, trace_kwargs=exit_trace_kwargs, fig=fig, **kwargs)
            else:
                fig = _self.new_entries.vbt.signals.plot_as_entry_markers(
                    y=entry_y, trace_kwargs=entry_trace_kwargs, fig=fig, **kwargs)
                fig = _self.exits.vbt.signals.plot_as_exit_markers(
                    y=exit_y, trace_kwargs=exit_trace_kwargs, fig=fig, **kwargs)

            return fig

        plot.__doc__ = """Plot `{0}.{1}` and `{0}.exits`.

        Args:
            entry_y (array_like): Y-axis values to plot entry markers on.
            exit_y (array_like): Y-axis values to plot exit markers on.
            entry_types (array_like): Entry types in string format.
            exit_types (array_like): Exit types in string format.
            entry_trace_kwargs (dict): Keyword arguments passed to \
            `vectorbt.signals.accessors.SignalsSRAccessor.plot_as_entry_markers` for `{0}.{1}`.
            exit_trace_kwargs (dict): Keyword arguments passed to \
            `vectorbt.signals.accessors.SignalsSRAccessor.plot_as_exit_markers` for `{0}.exits`.
            fig (Figure or FigureWidget): Figure to add traces to.
            **kwargs: Keyword arguments passed to `vectorbt.signals.accessors.SignalsSRAccessor.plot_as_markers`.
        """.format(
            self.class_name, 'new_entries' if mode == FactoryMode.Chain else 'entries'
        )

        setattr(self.Indicator, 'plot', plot)
```

## from_choice_func
基于入场和出场选择函数构建信号生成器类。
- `MySignalFactory().from_choice_func(...)`
  - 最后涉及 `self.from_custom_func(...)`，从而生成一个 `IndicatorBase` 实例
- `.run(...)`

### 参数

- `entry_choice_func` (callable): 返回入场信号索引的 `choice_func_nb` 函数
    - 对于 `FactoryMode.Chain` 模式，默认为 `vectorbt.signals.nb.first_choice_nb`
    - 该函数应该使用 @njit 装饰器进行Numba编译
    - 函数签名：`def func(from_i, to_i, col, *args) -> np.ndarray`
- `exit_choice_func` (callable): 返回出场信号索引的 `choice_func_nb` 函数
    - 该函数应该使用 @njit 装饰器进行Numba编译
    - 函数签名：`def func(from_i, to_i, col, *args) -> np.ndarray`
- `generate_func` (callable): 入场信号生成函数
    - 默认为 `vectorbt.signals.nb.generate_nb`
    - 负责根据选择函数的结果生成实际的入场信号
- `generate_ex_func` (callable): 出场信号生成函数
    - 默认为 `vectorbt.signals.nb.generate_ex_nb`
    - 负责根据选择函数的结果生成实际的出场信号
- `generate_enex_func` (callable): 入场和出场信号同时生成函数
    - 默认为 `vectorbt.signals.nb.generate_enex_nb`
    - 用于同时生成入场和出场信号的情况
- `cache_func` (callable): 缓存函数，用于预处理数据
    - 所有返回的对象将作为最后几个参数传递给选择函数
    - 用于提高重复计算的效率
- `entry_settings` (dict): 入场选择函数的设置字典
    - 控制哪些输入、参数和参数传递给入场函数
    - 详见下面的设置字典说明
- `exit_settings` (dict): 出场选择函数的设置字典
    - 控制哪些输入、参数和参数传递给出场函数
    - 详见下面的设置字典说明
- `cache_settings` (dict): 缓存函数的设置字典
    - 控制缓存函数的参数传递
    - 详见下面的设置字典说明
- `numba_loop` (bool): 是否使用Numba进行循环
    - 当对小型输入进行大量迭代时设置为True
    - 可以提高性能但会增加编译时间
- `**kwargs`: 传递给 `IndicatorFactory.from_custom_func` 的关键字参数

返回：`IndicatorBase` 类型实例

#### 设置字典 `entry/exit/cache_settings` 可以包含的键
- `pass_inputs` (list of str): 要传递给选择函数的输入名称列表
  - 默认为 []。顺序很重要。每个名称必须在 `input_names` 中
  - 示例：['close', 'volume'] 会将close和volume数据传递给函数
- `pass_in_outputs` (list of str): 要传递给选择函数的就地输出名称列表
  - 默认为 []。顺序很重要。每个名称必须在 `in_output_names` 中
  - 用于在函数间共享状态信息
- `pass_params` (list of str): 要传递给选择函数的参数名称列表
  - 默认为 []。顺序很重要。每个名称必须在 `param_names` 中
  - 示例：['threshold', 'period'] 会将阈值和周期参数传递给函数
- `pass_kwargs` (dict, list of str or list of tuple): 从 `kwargs` 字典中要作为位置参数传递给选择函数的关键字参数
  - 默认为 []。顺序很重要
  - 如果任何元素是元组，应包含名称和默认值
  - 如果任何元素是字符串，默认值为None

  内置键包括：
  - `input_shape`: 如果没有传递输入时间序列，则为输入形状
      - 如果 `pass_input_shape` 为True，则由管道提供默认值
  - `wait`: 放置信号前等待的刻度数
      - 默认为1
  - `until_next`: 是否将信号放置到下一个入场信号
      - 默认为True，仅在 `generate_ex_func` 中应用
  - `skip_until_exit`: 是否跳过处理入场信号直到下一个出场
      - 默认为False，仅在 `generate_ex_func` 中应用
  - `pick_first`: 是否在找到第一个出场信号时立即停止
      - 对于 `FactoryMode.Entries` 默认为False，否则为True
  - `temp_idx_arr`: 用于临时存储索引的空整数数组
      - 默认为自动生成的形状为 `input_shape[0]` 的数组
      - 也可以传递 `temp_idx_arr1`, `temp_idx_arr2` 等来生成多个
  - `flex_2d`: 参见 `vectorbt.base.reshape_fns.flex_select_auto_nb`
      - 如果 `pass_flex_2d` 为True，则由管道提供默认值
- `pass_cache` (bool): 是否将缓存从 `cache_func` 传递给选择函数
  - 默认为False。缓存以解包形式传递

#### 可以传递给 `run` 和 `run_combs` 方法的参数
- `*args`: 对于 `FactoryMode.Entries` 应使用此参数代替 `entry_args`，
  - 对于 `FactoryMode.Exits` 和默认 `entry_choice_func` 的 `FactoryMode.Chain` 应使用此参数代替 `exit_args`
- `entry_args` (tuple): 传递给入场选择函数的参数
- `exit_args` (tuple): 传递给出场选择函数的参数
- `cache_args` (tuple): 传递给缓存函数的参数
- `entry_kwargs` (tuple): 入场选择函数的设置。如果 `pass_kwargs` 中有参数，也包含作为位置参数传递的参数
- `exit_kwargs` (tuple): 出场选择函数的设置。如果 `pass_kwargs` 中有参数，也包含作为位置参数传递的参数
- `cache_kwargs` (tuple): 缓存函数的设置。如果 `pass_kwargs` 中有参数，也包含作为位置参数传递的参数
- `return_cache` (bool): 是否仅返回缓存
- `use_cache` (any): 要使用的缓存
- `**kwargs`: 对于 `FactoryMode.Entries` 应使用此参数代替 `entry_kwargs`，
  - 对于 `FactoryMode.Exits` 和默认 `entry_choice_func` 的 `FactoryMode.Chain` 应使用此参数代替 `exit_kwargs`

### 例子

#### 1 定义技术指标计算函数

In [ ]:
import numpy as np
import pandas as pd
from numba import njit
from vectorbt.signals.factory import SignalFactory
from vectorbt.signals.nb import first_choice_nb
import vectorbt as vbt

# ==================== 第一步：定义技术指标计算函数 ====================

@njit
def calculate_macd(close, fast_period=12, slow_period=26, signal_period=9):
    """计算MACD指标，支持2D输入"""
    # close: (n, m) ndarray
    n, m = close.shape
    fast_ema = np.empty((n, m), dtype=np.float64)
    slow_ema = np.empty((n, m), dtype=np.float64)
    # 初始化EMA
    for col in range(m):
        fast_ema[0, col] = close[0, col]
        slow_ema[0, col] = close[0, col]
    # 计算EMA
    fast_alpha = 2.0 / (fast_period + 1)
    slow_alpha = 2.0 / (slow_period + 1)
    for col in range(m):
        for i in range(1, n):
            fast_ema[i, col] = fast_alpha * close[i, col] + (1 - fast_alpha) * fast_ema[i-1, col]
            slow_ema[i, col] = slow_alpha * close[i, col] + (1 - slow_alpha) * slow_ema[i-1, col]
    # 计算MACD线
    macd_line = fast_ema - slow_ema
    # 计算信号线
    signal_line = np.empty((n, m), dtype=np.float64)
    for col in range(m):
        signal_line[0, col] = macd_line[0, col]
    signal_alpha = 2.0 / (signal_period + 1)
    for col in range(m):
        for i in range(1, n):
            signal_line[i, col] = signal_alpha * macd_line[i, col] + (1 - signal_alpha) * signal_line[i-1, col]
    return macd_line, signal_line

@njit
def calculate_rsi(close, period=14):
    """计算RSI指标，支持2D输入"""
    n, m = close.shape
    rsi = np.empty((n, m), dtype=np.float64)
    # 初始化前period个值为50
    for col in range(m):
        for i in range(period):
            rsi[i, col] = 50.0
    # 计算RSI
    for col in range(m):
        for i in range(period, n):
            gains = 0.0
            losses = 0.0
            for j in range(i-period+1, i+1):
                if j > 0:
                    change = close[j, col] - close[j-1, col]
                    if change > 0:
                        gains += change
                    else:
                        losses -= change
            avg_gain = gains / period
            avg_loss = losses / period
            if avg_loss == 0.0:
                rsi[i, col] = 100.0
            else:
                rs = avg_gain / avg_loss
                rsi[i, col] = 100.0 - (100.0 / (1.0 + rs))
    return rsi

#### 2 定义入场选择函数

In [ ]:
# ==================== 第二步：定义入场选择函数 ====================

@njit
def smart_entry_choice_nb(from_i, to_i, col, close, macd_line, signal_line, rsi, consecutive_losses, max_losses):
    """
    智能入场选择函数
    
    入场条件：
    1. MACD金叉（MACD线上穿信号线）
    2. RSI < 30（超卖反弹）
    3. 连续亏损次数 < max_losses（风险控制）
    """
    # 检查连续亏损次数
    if consecutive_losses[from_i, col] >= max_losses:
        return np.empty(0, dtype=np.int64)
    
    # 检查MACD金叉
    if from_i > 0:
        prev_macd = macd_line[from_i-1, col]
        prev_signal = signal_line[from_i-1, col]
        curr_macd = macd_line[from_i, col]
        curr_signal = signal_line[from_i, col]
        
        # MACD金叉：之前MACD线在信号线下方，现在在上方
        macd_crossover = (prev_macd <= prev_signal) and (curr_macd > curr_signal)
        
        # RSI超卖反弹
        rsi_oversold = rsi[from_i, col] < 30
        
        # 同时满足两个条件
        if macd_crossover and rsi_oversold:
            return np.array([from_i], dtype=np.int64)
    
    return np.empty(0, dtype=np.int64)

#### 3 定义出场选择函数

In [ ]:
# ==================== 第三步：定义出场选择函数 ====================

@njit
def smart_exit_choice_nb(from_i, to_i, col, close, macd_line, signal_line, rsi, entry_price, stop_loss_pct, take_profit_pct):
    """
    智能出场选择函数
    
    出场条件：
    1. MACD死叉（MACD线下穿信号线）
    2. RSI > 70（超买回调）
    3. 止损：价格下跌超过stop_loss_pct
    4. 止盈：价格上涨超过take_profit_pct
    """
    if from_i > 0:
        # 检查MACD死叉
        prev_macd = macd_line[from_i-1, col]
        prev_signal = signal_line[from_i-1, col]
        curr_macd = macd_line[from_i, col]
        curr_signal = signal_line[from_i, col]
        
        # MACD死叉：之前MACD线在信号线上方，现在在下方
        macd_crossunder = (prev_macd >= prev_signal) and (curr_macd < curr_signal)
        
        # RSI超买回调
        rsi_overbought = rsi[from_i, col] > 70
        
        # 技术指标出场条件
        technical_exit = macd_crossunder and rsi_overbought
        
        # 价格出场条件
        current_price = close[from_i, col]
        entry_p = entry_price[from_i, col]
        
        # 检查entry_price是否为NaN
        if np.isnan(entry_p):
            return np.empty(0, dtype=np.int64)
        
        price_change = (current_price - entry_p) / entry_p
        
        # 止损或止盈
        stop_loss = price_change < -stop_loss_pct
        take_profit = price_change > take_profit_pct
        
        # 满足任一出场条件
        if technical_exit or stop_loss or take_profit:
            return np.array([from_i], dtype=np.int64)
    
    return np.empty(0, dtype=np.int64)

#### 4 创建信号工厂

In [ ]:
# ==================== 第四步：创建信号工厂 ====================

# 创建智能交易信号工厂
# 注意：param_names只包含真正要传递给choice_func的参数
SmartTradingSignals = SignalFactory(
    mode='both',
    input_names=['close', 'macd_line', 'signal_line', 'rsi', 'consecutive_losses'],
    param_names=['max_losses', 'stop_loss_pct', 'take_profit_pct'],
    in_output_names=['entry_price'],
    attr_settings=dict(
        entry_price=dict(dtype=np.float64)
    )
).from_choice_func(
    entry_choice_func=smart_entry_choice_nb,
    exit_choice_func=smart_exit_choice_nb,
    entry_settings=dict(
        pass_inputs=['close', 'macd_line', 'signal_line', 'rsi', 'consecutive_losses'],
        pass_params=['max_losses']
    ),
    exit_settings=dict(
        pass_inputs=['close', 'macd_line', 'signal_line', 'rsi'],
        pass_in_outputs=['entry_price'],
        pass_params=['stop_loss_pct', 'take_profit_pct']
    )
)

#### 5 准备测试数据

In [ ]:
# ==================== 第五步：准备测试数据 ====================

# 生成模拟价格数据（模拟真实市场波动）
np.random.seed(42)
n_days = 252  # 一年的交易日
n_assets = 3  # 3只股票

# 生成价格数据
prices = np.empty((n_days, n_assets))
for col in range(n_assets):
    # 初始价格
    price = 100 + col * 10
    
    for i in range(n_days):
        # 添加随机波动
        daily_return = np.random.normal(0.001, 0.02)  # 日收益率
        price *= (1 + daily_return)
        prices[i, col] = price

# 计算技术指标
print("计算MACD指标...")
macd_line, signal_line = calculate_macd(prices, 12, 26, 9)

print("计算RSI指标...")
rsi = calculate_rsi(prices, 14)

# 初始化连续亏损计数
consecutive_losses = np.zeros_like(prices, dtype=np.int64)

#### 6 运行信号生成器

In [ ]:
# ==================== 第六步：运行信号生成器 ====================

# 设置参数 - 只包含要传递给choice_func的参数
params = dict(
    max_losses=3,
    stop_loss_pct=0.05,
    take_profit_pct=0.15
)

print("运行信号生成器...")
# 运行信号生成器
signals = SmartTradingSignals.run(
    close=prices,
    macd_line=macd_line,
    signal_line=signal_line,
    rsi=rsi,
    consecutive_losses=consecutive_losses,
    input_shape=prices.shape,  # 明确指定输入形状
    **params
)

#### 7 分析结果

In [ ]:
# ==================== 第七步：分析结果 ====================

print("=== 智能交易信号系统分析结果 ===")
print(f"数据期间: {n_days} 个交易日")
print(f"分析资产: {n_assets} 只股票")
print(f"参数设置: {params}")

# 转换为 numpy 数组
entries_array = signals.entries.values
exits_array = signals.exits.values

# 统计信号数量
total_entries = np.sum(entries_array)
total_exits = np.sum(exits_array)

print(f"\n信号统计:")
print(f"入场信号总数: {total_entries}")
print(f"出场信号总数: {total_exits}")

# 按资产统计
for col in range(n_assets):
    asset_entries = np.sum(entries_array[:, col])
    asset_exits = np.sum(exits_array[:, col])
    print(f"资产 {col+1}: {asset_entries} 个入场信号, {asset_exits} 个出场信号")

def calculate_returns(entries, exits, prices):
    """计算交易收益率"""
    # 确保输入是 numpy 数组
    if hasattr(entries, 'values'):
        entries = entries.values
    if hasattr(exits, 'values'):
        exits = exits.values
    
    returns = []
    for col in range(entries.shape[1]):
        for i in range(entries.shape[0]):
            if entries[i, col]:  # 入场信号
                entry_price = prices[i, col]
                # 找到对应的出场信号
                for j in range(i+1, entries.shape[0]):
                    if exits[j, col]:  # 出场信号
                        exit_price = prices[j, col]
                        trade_return = (exit_price - entry_price) / entry_price
                        returns.append(trade_return)
                        break
    return np.array(returns)

trade_returns = calculate_returns(signals.entries, signals.exits, prices)

if len(trade_returns) > 0:
    print(f"\n交易统计:")
    print(f"总交易次数: {len(trade_returns)}")
    print(f"平均收益率: {np.mean(trade_returns):.4f} ({np.mean(trade_returns)*100:.2f}%)")
    print(f"收益率标准差: {np.std(trade_returns):.4f}")
    print(f"最大单笔收益: {np.max(trade_returns):.4f} ({np.max(trade_returns)*100:.2f}%)")
    print(f"最大单笔亏损: {np.min(trade_returns):.4f} ({np.min(trade_returns)*100:.2f}%)")
    print(f"盈利交易比例: {np.sum(trade_returns > 0) / len(trade_returns):.2%}")
else:
    print("\n没有产生任何交易信号")

#### 8 可视化结果

In [ ]:
# ==================== 第八步：可视化结果 ====================

import matplotlib.pyplot as plt

# 选择第一只股票进行可视化
asset_idx = 0

# 获取信号数据 - 使用正确的方式
entries_array = signals.entries.values
exits_array = signals.exits.values

fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# 绘制价格和信号
axes[0].plot(prices[:, asset_idx], label='价格', color='blue', alpha=0.7)

# 检查是否有入场信号
if np.any(entries_array[:, asset_idx]):
    entry_indices = np.where(entries_array[:, asset_idx])[0]
    entry_prices = prices[entry_indices, asset_idx]
    axes[0].scatter(entry_indices, entry_prices, 
                    color='green', marker='^', s=100, label='入场信号', zorder=5)

# 检查是否有出场信号
if np.any(exits_array[:, asset_idx]):
    exit_indices = np.where(exits_array[:, asset_idx])[0]
    exit_prices = prices[exit_indices, asset_idx]
    axes[0].scatter(exit_indices, exit_prices, 
                    color='red', marker='v', s=100, label='出场信号', zorder=5)

axes[0].set_title(f'资产 {asset_idx+1} - 价格和交易信号')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 绘制MACD
axes[1].plot(macd_line[:, asset_idx], label='MACD线', color='blue')
axes[1].plot(signal_line[:, asset_idx], label='信号线', color='orange')
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1].set_title('MACD指标')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 绘制RSI
axes[2].plot(rsi[:, asset_idx], label='RSI', color='purple')
axes[2].axhline(y=70, color='red', linestyle='--', alpha=0.7, label='超买线(70)')
axes[2].axhline(y=30, color='green', linestyle='--', alpha=0.7, label='超卖线(30)')
axes[2].set_title('RSI指标')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### 9 策略优化示例

In [ ]:
# ==================== 第九步：策略优化示例 ====================

print("\n=== 策略参数优化 ===")

# 测试不同的参数组合
param_combinations = [
    {'max_losses': 2, 'stop_loss_pct': 0.03, 'take_profit_pct': 0.10},
    {'max_losses': 3, 'stop_loss_pct': 0.05, 'take_profit_pct': 0.15},
    {'max_losses': 4, 'stop_loss_pct': 0.07, 'take_profit_pct': 0.20},
]

for i, params in enumerate(param_combinations):
    print(f"\n参数组合 {i+1}: {params}")
    
    # 运行信号生成器
    signals_opt = SmartTradingSignals.run(
        close=prices,
        macd_line=macd_line,
        signal_line=signal_line,
        rsi=rsi,
        consecutive_losses=consecutive_losses,
        input_shape=prices.shape,
        **params
    )
    
    # 计算收益率
    trade_returns_opt = calculate_returns(signals_opt.entries, signals_opt.exits, prices)
    
    if len(trade_returns_opt) > 0:
        total_return = np.sum(trade_returns_opt)
        avg_return = np.mean(trade_returns_opt)
        win_rate = np.sum(trade_returns_opt > 0) / len(trade_returns_opt)
        
        print(f"  总交易次数: {len(trade_returns_opt)}")
        print(f"  总收益率: {total_return:.4f} ({total_return*100:.2f}%)")
        print(f"  平均收益率: {avg_return:.4f} ({avg_return*100:.2f}%)")
        print(f"  胜率: {win_rate:.2%}")
    else:
        print("  没有产生交易信号")